# Test Granite Tiny Polish Capabilities

**How to use this notebook:**
1. Go to [Google Colab](https://colab.research.google.com)
2. Click File → Upload notebook
3. Upload this file: `06-test_granite_polish.ipynb`
4. Select Runtime → Change runtime type → GPU (T4/V100/A100)
5. Run all cells

This notebook tests Granite Tiny model on Polish language tasks **before and after fine-tuning**.

**Usage:**
1. Run this notebook BEFORE fine-tuning to get baseline results
2. Fine-tune the model using `granite_tiny_polish_finetuning_pro.ipynb`
3. Run this notebook AFTER fine-tuning to compare results

**Requirements:**
- Google Colab (Free or Pro)
- GPU runtime (T4, V100, or A100)

## Step 1: Check GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 2: Install Dependencies

In [ ]:
!pip install --upgrade --no-cache-dir --quiet unsloth
!pip install --upgrade --quiet transformers accelerate bitsandbytes

print("✓ Installation complete!")

## Step 3: Upload Test Questions

Upload the `polish_test_questions.jsonl` file from your local machine.

In [ ]:
from google.colab import files

print("Please upload polish_test_questions.jsonl:")
uploaded = files.upload()

if 'polish_test_questions.jsonl' in uploaded:
    print("✓ File uploaded successfully!")
    !wc -l polish_test_questions.jsonl
else:
    print("⚠ Please upload a file named 'polish_test_questions.jsonl'")

## Step 4: Load Test Questions

In [ ]:
import json

def load_test_questions(filepath="polish_test_questions.jsonl"):
    """Load test questions from JSONL file."""
    questions = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            questions.append(json.loads(line))
    return questions

questions = load_test_questions()
print(f"✓ Loaded {len(questions)} test questions")

# Show categories
categories = set(q['category'] for q in questions)
print(f"\nCategories: {', '.join(sorted(categories))}")

## Step 5: Configure Model to Test

**Choose ONE option:**

### Option A: Test Base Model (Before Fine-tuning)
Run this to get baseline results before fine-tuning.

In [ ]:
# Option A: Base model
MODEL_NAME = "unsloth/granite-4.0-h-tiny"
OUTPUT_FILE = "results_before_finetuning.json"

print(f"Testing: {MODEL_NAME}")
print(f"Results will be saved to: {OUTPUT_FILE}")

### Option B: Test Fine-tuned Model (After Fine-tuning)

**First**, upload your fine-tuned model directory or load from Google Drive.

Then run this cell:

In [ ]:
# Option B: Fine-tuned model
# If you saved to Google Drive:
from google.colab import drive
drive.mount('/content/drive')

MODEL_NAME = "/content/drive/MyDrive/granite4-tiny-h-polish-lora-pro"
# Or if uploaded locally:
# MODEL_NAME = "granite4-tiny-h-polish-lora-pro"

OUTPUT_FILE = "results_after_finetuning.json"

print(f"Testing: {MODEL_NAME}")
print(f"Results will be saved to: {OUTPUT_FILE}")

## Step 6: Load Model

In [ ]:
from unsloth import FastLanguageModel

print(f"Loading model: {MODEL_NAME}")
print("This may take 2-3 minutes...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Enable inference mode
FastLanguageModel.for_inference(model)

print("✓ Model loaded successfully!")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 7: Define Test Functions

In [ ]:
def format_prompt(instruction, input_text=""):
    """Format prompt using Granite's chat template."""
    user_message = instruction
    if input_text:
        user_message += f"\n\n{input_text}"
    
    prompt = f"""<|start_of_role|>user<|end_of_role|>
{user_message}
<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>
"""
    return prompt


def generate_response(prompt, max_new_tokens=512, temperature=0.7):
    """Generate response from the model."""
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract assistant response
    assistant_start = response.find("<|start_of_role|>assistant<|end_of_role|>")
    if assistant_start != -1:
        assistant_response = response[assistant_start + len("<|start_of_role|>assistant<|end_of_role|>"):].strip()
        assistant_response = assistant_response.replace("<|end_of_text|>", "").strip()
        return assistant_response
    
    return response


def test_single_question(question, show_details=True):
    """Test model on a single question."""
    if show_details:
        print(f"\n{'='*60}")
        print(f"ID: {question['id']} | Category: {question['category']}")
        print(f"{'='*60}")
        print(f"Instruction: {question['instruction']}")
        if question['input']:
            print(f"Input: {question['input'][:100]}..." if len(question['input']) > 100 else f"Input: {question['input']}")
    
    # Format and generate
    prompt = format_prompt(question['instruction'], question['input'])
    
    try:
        response = generate_response(prompt)
        
        if show_details:
            print(f"\nExpected: {question['expected_output'][:150]}..." if len(question['expected_output']) > 150 else f"\nExpected: {question['expected_output']}")
            print(f"\nModel Output:\n{response}")
        
        return {
            "id": question['id'],
            "category": question['category'],
            "instruction": question['instruction'],
            "input": question['input'],
            "expected_output": question['expected_output'],
            "model_output": response,
            "success": True
        }
    except Exception as e:
        if show_details:
            print(f"\n⚠ Error: {str(e)}")
        return {
            "id": question['id'],
            "category": question['category'],
            "instruction": question['instruction'],
            "input": question['input'],
            "expected_output": question['expected_output'],
            "model_output": f"ERROR: {str(e)}",
            "success": False
        }

print("✓ Test functions defined!")

## Step 8: Run Tests

This will test all 20 questions. Takes ~5-10 minutes.

In [ ]:
from datetime import datetime

print(f"\n{'='*60}")
print(f"TESTING: {MODEL_NAME}")
print(f"{'='*60}")
print(f"Total questions: {len(questions)}")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

results = []
for i, question in enumerate(questions, 1):
    print(f"\n[{i}/{len(questions)}] Testing...")
    result = test_single_question(question, show_details=True)
    results.append(result)

print(f"\n{'='*60}")
print("✓ Testing complete!")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

## Step 9: Analyze Results

In [ ]:
# Summary statistics
total = len(results)
successful = sum(1 for r in results if r['success'])
failed = total - successful

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total questions: {total}")
print(f"Successful: {successful} ({successful/total*100:.1f}%)")
print(f"Failed: {failed} ({failed/total*100:.1f}%)")

# By category
print(f"\n{'='*60}")
print("BY CATEGORY")
print(f"{'='*60}")

categories = {}
for r in results:
    cat = r['category']
    if cat not in categories:
        categories[cat] = {'total': 0, 'success': 0}
    categories[cat]['total'] += 1
    if r['success']:
        categories[cat]['success'] += 1

for cat in sorted(categories.keys()):
    stats = categories[cat]
    rate = (stats['success'] / stats['total']) * 100
    print(f"{cat:20s}: {stats['success']}/{stats['total']} ({rate:.0f}%)")

## Step 10: Save Results

In [ ]:
output_data = {
    "model": MODEL_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_questions": total,
    "successful": successful,
    "failed": failed,
    "success_rate": successful / total,
    "results": results
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\n✓ Results saved to: {OUTPUT_FILE}")
print(f"\nFile size: {len(json.dumps(output_data, ensure_ascii=False)) / 1024:.1f} KB")

## Step 11: Download Results

In [ ]:
from google.colab import files

files.download(OUTPUT_FILE)
print(f"✓ Downloaded: {OUTPUT_FILE}")

## Step 12: Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Copy to Drive
drive_path = f"/content/drive/MyDrive/{OUTPUT_FILE}"
shutil.copy(OUTPUT_FILE, drive_path)

print(f"✓ Saved to Google Drive: {drive_path}")

## Next Steps

### If you tested the BASE model:
1. Download `results_before_finetuning.json`
2. Fine-tune the model using `granite_tiny_polish_finetuning_pro.ipynb`
3. Come back and test the fine-tuned model

### If you tested the FINE-TUNED model:
1. Download `results_after_finetuning.json`
2. Compare with `results_before_finetuning.json`
3. Analyze improvements in:
   - Polish grammar and fluency
   - Task-specific accuracy
   - Instruction following
   - Response quality

### Comparison Tips:
- Look at specific examples side-by-side
- Check if Polish responses are more natural
- Verify task completion accuracy
- Note any regressions or improvements